# Civil War News — Notebook 3c: Representation Update

Refines BERTopic topic keyword representations on an already-fitted model.
Runs on **local machine or Colab** — no GPU required for rep-only modes.

## Modes

| `REVECTORIZE` | `REPRESENTATION_MODE` | Needs news_proc? | Recommended platform |
|---|---|---|---|
| False | `mmr` | No | Local (seconds) |
| False | `keybert` | No | Local (~3 min CPU) |
| False | `zeroshot` | No | Local (~15 min CPU) |
| False | `textgen_small` | No | Local (~10 min CPU) |
| True | any | Yes (from HF or Drive) | Colab (fast) or local (slow) |

**First-time setup:** add `HF_TOKEN` to Colab Secrets or set it as an environment variable locally.

In [ ]:
%pip install -q bertopic sentence-transformers huggingface_hub datasets
print('Packages ready.')

In [ ]:
import os, zipfile
import numpy as np
from bertopic import BERTopic
from bertopic.representation import (
    KeyBERTInspired,
    MaximalMarginalRelevance,
    ZeroShotClassification,
    TextGeneration,
)
from huggingface_hub import HfApi, login, hf_hub_download
from sentence_transformers import SentenceTransformer
from sentence_transformers.sentence_transformer import modules as ST_models

try:
    _IN_COLAB = True
    from google.colab import userdata as _ud
except ImportError:
    _IN_COLAB = False
    _ud = None

print(f'Running in {"Colab" if _IN_COLAB else "local environment"}')

## Section 2: Configuration

In [ ]:
# ── Edit these ────────────────────────────────────────────────────────────────
MODEL_NAME = 'macberth'   # 'macberth'  or  'bert_1850_1875'

# Representation mode — choose one:
#   'mmr'          MaximalMarginalRelevance — no embedding model needed, seconds
#   'keybert'      KeyBERTInspired + MMR — needs embedding model, ~3 min CPU
#   'zeroshot'     ZeroShotClassification — custom labels, ~15 min CPU
#   'textgen_small' TextGeneration with Flan-T5-small — ~10 min CPU
#   'openai'       OpenAI API — set OPENAI_API_KEY, costs ~$0.01 for 4415 topics
REPRESENTATION_MODE = 'keybert'

# Zero-shot labels (only used when REPRESENTATION_MODE = 'zeroshot')
ZEROSHOT_LABELS = [
    'slavery and bondage',
    'abolition and emancipation',
    'elections and political parties',
    'civil war military',
    'economics and commerce',
    'literature and culture',
]

# Re-vectorize: rebuild c-TF-IDF vocabulary from a document sample.
# False = only update keyword representations (fast, no news_proc needed).
# True  = also rebuild word lists from documents (slower, needs news_proc).
REVECTORIZE = False

# Sample size for re-vectorization (only used when REVECTORIZE=True).
# 500_000 works well locally; 2_000_000 recommended on Colab A100.
REVECTORIZE_SAMPLE = 500_000

# Storage
HF_MODEL_REPO = 'patrickjcrawford/civil-war-news'   # BERTopic model source + dest
HF_NEWS_REPO  = 'patrickjcrawford/civil-war-news'   # news_proc.zip (if REVECTORIZE=True)
LOCAL_BASE    = r'C:\Users\Patrick\Documents\Projects\Dissertation\Civil-War-News\data\BERTopic'
DRIVE_BASE    = '/content/drive/MyDrive/CivilWarNews'   # Colab only
# ─────────────────────────────────────────────────────────────────────────────

_MODEL_IDS = {
    'macberth':       'emanjavacas/MacBERTh',
    'bert_1850_1875': 'Livingwithmachines/bert_1850_1875',
}
MODEL_ID = _MODEL_IDS[MODEL_NAME]

print(f'Model      : {MODEL_NAME} ({MODEL_ID})')
print(f'Rep mode   : {REPRESENTATION_MODE}')
print(f'Revectorize: {REVECTORIZE}')
print(f'Platform   : {"Colab" if _IN_COLAB else "local"}')

## Section 3: Auth & Storage

In [ ]:
# ── Hugging Face login ─────────────────────────────────────────────────────
try:
    _token = _ud.get('HF_TOKEN') if _ud else os.environ.get('HF_TOKEN')
    login(token=_token, add_to_git_credential=False)
    print('[OK] Logged in to Hugging Face.')
except Exception as _e:
    print(f'HF login: {_e}')
    print('Run login() interactively or set HF_TOKEN env var / Colab Secret.')

# ── Drive mount (Colab only) ───────────────────────────────────────────────
_NEWS_PROC_PATH = os.path.join(LOCAL_BASE, '..', 'news_proc')
if _IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    os.makedirs('/content/data', exist_ok=True)
    _NEWS_PROC_PATH = '/content/data/news_proc'
    print('[OK] Drive mounted.')

## Section 4: Load BERTopic Model

In [ ]:
_pkl = os.path.join(LOCAL_BASE, f'bertopic_model_{MODEL_NAME}.pkl')
_zip = os.path.join(LOCAL_BASE, f'bertopic_model_{MODEL_NAME}.zip')
_dir = os.path.join(LOCAL_BASE, f'bertopic_model_{MODEL_NAME}')

if os.path.exists(_pkl):
    print(f'Loading pickle: {_pkl}')
    topic_model = BERTopic.load(_pkl)
elif os.path.exists(_zip):
    print(f'Extracting zip: {_zip}')
    with zipfile.ZipFile(_zip, 'r') as _zf:
        _zf.extractall(LOCAL_BASE)
    topic_model = BERTopic.load(_dir)
elif os.path.isdir(_dir):
    print(f'Loading safetensors dir: {_dir}')
    topic_model = BERTopic.load(_dir)
else:
    print(f'Model not found locally. Downloading from HF ({HF_MODEL_REPO}/{MODEL_NAME}/) ...')
    _zip_hf = hf_hub_download(repo_id=HF_MODEL_REPO,
                              filename=f'{MODEL_NAME}/bertopic_model_{MODEL_NAME}.zip',
                              repo_type='dataset')
    with zipfile.ZipFile(_zip_hf, 'r') as _zf:
        _zf.extractall(LOCAL_BASE)
    topic_model = BERTopic.load(_dir)

_n_topics = len(topic_model.get_topic_info()) - 1
print(f'[OK] Model loaded: {MODEL_NAME}  ({_n_topics} topics)')

## Section 5: Load Documents

**`REVECTORIZE=False`:** Uses representative docs already stored inside the model (~3 per topic). No news_proc needed.

**`REVECTORIZE=True`:** Loads a sample from news_proc (Drive → HF fallback) and rebuilds vocabulary.

In [ ]:
docs, doc_topics = None, None

if REVECTORIZE:
    def _is_hf_dataset(path):
        return (os.path.exists(f'{path}/dataset_info.json') or
                os.path.exists(f'{path}/dataset_dict.json'))

    if not _is_hf_dataset(_NEWS_PROC_PATH):
        _drive_zip = f'{DRIVE_BASE}/news_proc.zip'
        if _IN_COLAB and os.path.exists(_drive_zip):
            print(f'Extracting news_proc.zip → /content/data ...')
            with zipfile.ZipFile(_drive_zip, 'r') as _zf:
                _zf.extractall('/content/data')
        else:
            if _IN_COLAB:
                print(f'Not found on Drive. Downloading from HF ({HF_NEWS_REPO}) ...')
            else:
                print(f'Downloading news_proc.zip from HF ({HF_NEWS_REPO}) ...')
            _dest = '/content/data' if _IN_COLAB else os.path.dirname(LOCAL_BASE)
            os.makedirs(_dest, exist_ok=True)
            _hf_zip = hf_hub_download(repo_id=HF_NEWS_REPO,
                                      filename='news_proc.zip', repo_type='dataset')
            with zipfile.ZipFile(_hf_zip, 'r') as _zf:
                _zf.extractall(_dest)
        print('Extracted.')

    from datasets import load_from_disk
    _news    = load_from_disk(_NEWS_PROC_PATH)
    _n       = min(REVECTORIZE_SAMPLE, len(_news))
    _news    = _news.shuffle(seed=42).select(range(_n))
    docs     = _news['article']
    print(f'Loaded {len(docs):,} articles')

    # Try to load topic assignments for the sample
    _topics_path = os.path.join(LOCAL_BASE, f'topics_{MODEL_NAME}.npy')
    if os.path.exists(_topics_path):
        _all_topics = np.load(_topics_path)
        doc_topics  = [int(_all_topics[i]) for i in range(len(docs))]
        print(f'Loaded {len(doc_topics):,} topic assignments')
    else:
        doc_topics = None
        print('topics.npy not found — passing docs without assignments')
else:
    # Use representative docs stored inside the model — no news_proc needed
    docs = [
        doc
        for topic_docs in topic_model.representative_docs_.values()
        for doc in topic_docs
    ]
    doc_topics = [
        t
        for t, topic_docs in topic_model.representative_docs_.items()
        for _ in topic_docs
    ]
    print(f'Using {len(docs):,} representative docs stored in model (REVECTORIZE=False)')

## Section 6: Build Representation Model

In [ ]:
# ── Load embedding model if needed ────────────────────────────────────────
_emb_model = None
if REPRESENTATION_MODE == 'keybert':
    print(f'Loading embedding model: {MODEL_ID} ...')
    _word_model    = ST_models.Transformer(MODEL_ID, max_seq_length=512)
    _pooling_model = ST_models.Pooling(
        _word_model.get_word_embedding_dimension(),
        pooling_mode_mean_tokens=True,
    )
    _emb_model = SentenceTransformer(modules=[_word_model, _pooling_model])
    print('Embedding model ready.')

# ── Build representation model ─────────────────────────────────────────────
if REPRESENTATION_MODE == 'mmr':
    representation_model = MaximalMarginalRelevance(diversity=0.3)

elif REPRESENTATION_MODE == 'keybert':
    representation_model = {
        'KeyBERT': KeyBERTInspired(embedding_model=_emb_model),
        'MMR':     MaximalMarginalRelevance(diversity=0.3),
    }

elif REPRESENTATION_MODE == 'zeroshot':
    representation_model = ZeroShotClassification(
        zeroshot_min_similarity=0.1,
        model='facebook/bart-large-mnli',
        candidate_topics=ZEROSHOT_LABELS,
    )

elif REPRESENTATION_MODE == 'textgen_small':
    from transformers import pipeline as _hf_pipe
    _gen = _hf_pipe('text2text-generation', model='google/flan-t5-small')
    representation_model = TextGeneration(
        _gen,
        prompt='I have a topic with these keywords: [KEYWORDS]. The topic name is:',
        diversity=0.1,
    )

elif REPRESENTATION_MODE == 'openai':
    import openai as _oai
    from bertopic.representation import OpenAI as BERTopicOpenAI
    representation_model = BERTopicOpenAI(
        client=_oai.OpenAI(),
        model='gpt-4o-mini',
        chat=True,
        prompt='I have a topic described by these keywords: [KEYWORDS]. Give a short (3-5 word) topic label.',
    )

else:
    raise ValueError(f'Unknown REPRESENTATION_MODE: {REPRESENTATION_MODE!r}')

print(f'Representation model ready: {REPRESENTATION_MODE}')

## Section 7: Update Topics

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import time as _time
import pandas as pd

_t0 = _time.time()

if REVECTORIZE:
    _vectorizer = CountVectorizer(
        stop_words='english',
        ngram_range=(1, 2),
        min_df=5,
        max_df=0.85,
        max_features=50_000,
    )
    topic_model.update_topics(
        docs,
        topics=doc_topics,
        vectorizer_model=_vectorizer,
        representation_model=representation_model,
        top_n_words=20,
    )
else:
    topic_model.update_topics(
        docs,
        topics=doc_topics,
        representation_model=representation_model,
        top_n_words=20,
    )

print(f'Done in {_time.time()-_t0:.0f}s')
pd.set_option('display.max_colwidth', 120)
topic_model.get_topic_info()[['Topic', 'Count', 'Name', 'Representation']].head(20)

## Section 8: Save Updated Model

In [ ]:
import json as _json
import numpy as _np

_orig = _json.JSONEncoder.default
def _np_default(self, obj):
    if isinstance(obj, _np.integer): return int(obj)
    if isinstance(obj, _np.floating): return float(obj)
    if isinstance(obj, _np.ndarray): return obj.tolist()
    return _orig(self, obj)
_json.JSONEncoder.default = _np_default

_save_dir = os.path.join(LOCAL_BASE, f'bertopic_model_{MODEL_NAME}')
topic_model.save(_save_dir, serialization='safetensors', save_ctfidf=True)

_json.JSONEncoder.default = _orig
print(f'Model saved locally: {_save_dir}')

# Zip
_zip_path = os.path.join(
    '/content' if _IN_COLAB else LOCAL_BASE,
    f'bertopic_model_{MODEL_NAME}.zip'
)
print('Zipping ...')
with zipfile.ZipFile(_zip_path, 'w', zipfile.ZIP_DEFLATED) as _zf:
    for _root, _, _fnames in os.walk(_save_dir):
        for _fname in _fnames:
            _fpath = os.path.join(_root, _fname)
            _zf.write(_fpath, os.path.relpath(_fpath, os.path.dirname(_save_dir)))

# Upload under MODEL_NAME subfolder (overwrites existing)
_api = HfApi()
_api.create_repo(HF_MODEL_REPO, private=True, repo_type='dataset', exist_ok=True)
_hf_name = f'{MODEL_NAME}/bertopic_model_{MODEL_NAME}.zip'
try:
    _existing = list(_api.list_repo_files(HF_MODEL_REPO, repo_type='dataset'))
except Exception:
    _existing = []
if _hf_name in _existing:
    _api.delete_file(_hf_name, HF_MODEL_REPO, repo_type='dataset')
    print(f'Deleted old {_hf_name} from HF.')
_api.upload_file(
    path_or_fileobj=_zip_path,
    path_in_repo=_hf_name,
    repo_id=HF_MODEL_REPO,
    repo_type='dataset',
)
print(f'Saved: hf://datasets/{HF_MODEL_REPO}/{_hf_name}')